# Ingesting and Validating Data

In [1]:
import os
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition\\prototype'

In [2]:
os.chdir('../')
%pwd

'c:\\Users\\uNUKE\\OneDrive\\Documents\\DSML\\GeoCognition'

In [3]:
import io
import json
from minio import Minio
from minio.error import S3Error
from dotenv import load_dotenv
from include.helpers import read_yaml
from include.constants import CONFIG_FILE_PATH
import pandas as pd
from include.constants import EQ_COLUMNS
import numpy as np

load_dotenv()

True

# Reading in Data From MinIO

In [5]:
# Connection settings
endpoint = "localhost:9000"  # Update if using a different host/portq
access_key = os.getenv("MINIO_ACCESS_KEY", "minio")
secret_key = os.getenv("MINIO_SECRET_KEY", "minio123")

client = Minio(
    endpoint,
    access_key=access_key,
    secret_key=secret_key,
    secure=False,
)

BUCKET_NAME = "earthquake-raw"
OBJECT_NAME = "year=2025/month=01/day=03/raw.json"

response = client.get_object(BUCKET_NAME, OBJECT_NAME)
try:
    payload = json.loads(response.read().decode("utf-8"))
finally:
    response.close()
    response.release_conn()

payload['features']

[{'type': 'Feature',
  'properties': {'mag': 4.3,
   'place': '22 km S of Āwash, Ethiopia',
   'time': 1735862540823,
   'updated': 1742247260040,
   'tz': None,
   'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/us6000pk69',
   'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000pk69&format=geojson',
   'felt': None,
   'cdi': None,
   'mmi': None,
   'alert': None,
   'status': 'reviewed',
   'tsunami': 0,
   'sig': 284,
   'net': 'us',
   'code': '6000pk69',
   'ids': ',us6000pk69,',
   'sources': ',us,',
   'types': ',origin,phase-data,',
   'nst': 23,
   'dmin': 1.507,
   'rms': 0.98,
   'gap': 112,
   'magType': 'mb',
   'type': 'earthquake',
   'title': 'M 4.3 - 22 km S of Āwash, Ethiopia'},
  'geometry': {'type': 'Point', 'coordinates': [40.2002, 8.7824, 10]},
  'id': 'us6000pk69'},
 {'type': 'Feature',
  'properties': {'mag': 2.5,
   'place': '147 km SSE of Old Harbor, Alaska',
   'time': 1735863443781,
   'updated': 1742247261040,
   'tz': None

## Must flatten the feature key

In [6]:
len(payload['features'])

79

In [7]:
payload

{'type': 'FeatureCollection',
 'metadata': {'generated': 1773743960000,
  'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2025-01-03T00%3A00%3A00&endtime=2025-01-04T00%3A00%3A00&minmagnitude=2.5&orderby=time-asc&limit=20000',
  'title': 'USGS Earthquakes',
  'status': 200,
  'api': '2.4.0',
  'limit': 20000,
  'offset': 1},
 'features': [{'type': 'Feature',
   'properties': {'mag': 4.3,
    'place': '22 km S of Āwash, Ethiopia',
    'time': 1735862540823,
    'updated': 1742247260040,
    'tz': None,
    'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/us6000pk69',
    'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000pk69&format=geojson',
    'felt': None,
    'cdi': None,
    'mmi': None,
    'alert': None,
    'status': 'reviewed',
    'tsunami': 0,
    'sig': 284,
    'net': 'us',
    'code': '6000pk69',
    'ids': ',us6000pk69,',
    'sources': ',us,',
    'types': ',origin,phase-data,',
    'nst': 23,
    'dmin': 

In [ ]:
# If some rows are JSON strings, parse them first.
def to_list(v) -> list:
    """
    Convert a JSON string to a list, or return the value as-is if it's not a string.
    """
    if isinstance(v, str):
        try:
            return json.loads(v)   # e.g. "[40.1, 9.2, 10.0]" -> [40.1, 9.2, 10.0]
        except Exception:
            return np.nan
    return v

In [16]:
df = pd.json_normalize(payload["features"])

coords = df["geometry.coordinates"].apply(to_list)

df["longitude"] = coords.str[0]
df["latitude"] = coords.str[1]
df["depth_km"] = coords.str[2]   # optional

df.drop(columns=["geometry.coordinates"], inplace=True)

# Remove only the properties prefix
df.columns

Index(['type', 'id', 'properties.mag', 'properties.place', 'properties.time',
       'properties.updated', 'properties.tz', 'properties.url',
       'properties.detail', 'properties.felt', 'properties.cdi',
       'properties.mmi', 'properties.alert', 'properties.status',
       'properties.tsunami', 'properties.sig', 'properties.net',
       'properties.code', 'properties.ids', 'properties.sources',
       'properties.types', 'properties.nst', 'properties.dmin',
       'properties.rms', 'properties.gap', 'properties.magType',
       'properties.type', 'properties.title', 'geometry.type', 'longitude',
       'latitude', 'depth_km'],
      dtype='object')

In [13]:
print(list(df_features.columns))

['type', 'id', 'properties.mag', 'properties.place', 'properties.time', 'properties.updated', 'properties.tz', 'properties.url', 'properties.detail', 'properties.felt', 'properties.cdi', 'properties.mmi', 'properties.alert', 'properties.status', 'properties.tsunami', 'properties.sig', 'properties.net', 'properties.code', 'properties.ids', 'properties.sources', 'properties.types', 'properties.nst', 'properties.dmin', 'properties.rms', 'properties.gap', 'properties.magType', 'properties.type', 'properties.title', 'geometry.type', 'geometry.coordinates']


In [32]:
len(df_features)

89

In [29]:
df_features.isnull().sum()

type            0
id              0
mag             0
place           0
time            0
updated         0
tz             89
url             0
detail          0
felt           69
cdi            69
mmi            84
alert          88
status          0
tsunami         0
sig             0
net             0
code            0
ids             0
sources         0
types           0
nst             5
dmin            5
rms             0
gap             5
magType         0
type            0
title           0
type            0
coordinates     0
dtype: int64

In [52]:
list(df_features.columns)

['type',
 'id',
 'mag',
 'place',
 'time',
 'updated',
 'tz',
 'url',
 'detail',
 'felt',
 'cdi',
 'mmi',
 'alert',
 'status',
 'tsunami',
 'sig',
 'net',
 'code',
 'ids',
 'sources',
 'types',
 'nst',
 'dmin',
 'rms',
 'gap',
 'magType',
 'type',
 'title',
 'type',
 'coordinates']

In [ ]:
import logging
logger = logging.getLogger(__name__)
logger.info("Hello World")